# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [1]:
# importar librerías
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import pointbiserialr
from scipy.stats import chi2_contingency
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
# cargar archivos
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')

In [3]:
# explorar datasets- entender como se componen los datasets
orders.head(10)

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28
5,order_5,user_2792,2025-04-22,mexico,desktop,organic,Tablet-Standard-64GB,Electronica,1.0,179.34,0.0,179.34
6,order_6,user_2802,2025-03-15,Colombia,mobile,organic,Blender-XL-Red,Hogar,2.0,163.59,0.0,327.19
7,order_7,user_1083,2025-05-01,Mexico,desktop,organic,Tablet-Standard-64GB,Electronica,2.0,373.68,0.0,747.35
8,order_8,user_2352,2025-03-01,Argentina,desktop,paid_search,Laptop-Gaming-16GB,Electronica,2.0,477.27,5.0,949.54
9,order_9,user_7001,2025-05-31,Mexico,mobile,organic,Sneakers-Urban-42,Moda,1.0,339.30,5.0,334.30


In [4]:
# Se le puede quitar users al id:usuario para que sea solo numeros, al igual que el numero de orden 
#(pero no es necesario, no camabia en nada)
#revisar duplicados
#corregir los nombres de mexico que hay diferentes
#hay columnas con un poco menos de información, hay valores ausentes pero es un porcentaje minimo lo cual me permite eliminarlos.
#valores numericos estan en formato correcto.
#cambiar fecha a tipo date
orders.info(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB


In [5]:
catalog.head(10)

,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"
5,Sneakers-Urban-42,Moda,17.21,Greene-Smith
6,Jacket-Winter-M,Moda,189.31,Mcmillan-Rhodes


In [6]:
catalog.info(10)
#no hay variables austentes

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes


In [7]:
marketing.head(10)

,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40
5,2025-01-01,Colombia,social_Colombia,social,833.38
6,2025-01-01,Argentina,organic_Argentina,organic,1365.62
7,2025-01-01,Argentina,paid_search_Argentina,paid_search,1282.65
8,2025-01-01,Argentina,social_Argentina,social,1534.69
9,2025-01-02,Mexico,organic_Mexico,organic,2537.38


In [8]:
marketing.info(10)
#Canal tiene un poco menos de variables, revisar y eliminar ausentes, que tienne un porcentaje minimo
#id_campaña es la union entre pais y canal
# se debe cambiar la fecha a tipo date

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas 

---

In [9]:
# Convertir Fechas a tipo Date para Orders
orders['fecha_hora_pedido']= pd.to_datetime(orders['fecha_hora_pedido']) 
orders.info() 


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           25100 non-null  object        
 1   id_usuario          25100 non-null  object        
 2   fecha_hora_pedido   25100 non-null  datetime64[ns]
 3   pais                24800 non-null  object        
 4   dispositivo         25080 non-null  object        
 5   fuente_referencia   25070 non-null  object        
 6   nombre_producto     25070 non-null  object        
 7   categoria_producto  25020 non-null  object        
 8   cantidad            25050 non-null  float64       
 9   precio_unitario     25050 non-null  float64       
 10  monto_descuento     25050 non-null  float64       
 11  monto_total         25100 non-null  float64       
dtypes: datetime64[ns](1), float64(4), object(7)
memory usage: 2.3+ MB


In [10]:
# Convertir Fechas a tipo Date para Marketing
marketing['fecha']= pd.to_datetime(marketing['fecha']) 
marketing.info() 


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   fecha       1620 non-null   datetime64[ns]
 1   pais        1620 non-null   object        
 2   id_campaña  1620 non-null   object        
 3   canal       1519 non-null   object        
 4   gasto       1620 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(3)
memory usage: 63.4+ KB


In [11]:
# Revisar los años y meses presentes.
print("orders:", orders['fecha_hora_pedido'].dt.year.unique(), orders['fecha_hora_pedido'].dt.month.unique()) 
print("marketing:", marketing['fecha'].dt.year.unique(), marketing['fecha'].dt.month.unique()) 

#mismos meses y años.


orders: [2025] [5 6 3 4 2 1]
marketing: [2025] [1 2 3 4 5 6]


In [12]:
#revisar variables numericas no tenga negativos para Orders
var_numerica =['cantidad', 'precio_unitario', 'monto_descuento', 'monto_total']
orders[var_numerica].describe()

#puedo encontrar sentinels en las variables numericas?
sentinels_orders = [-999, 0.0000, -1, 9999, "NA", "?", "NULL", "Unknown", "", " "]

for col in var_numerica:
    sentinel_count_orders = orders[col].isin(sentinels_orders).sum()
    if sentinel_count_orders > 0:
        print(f"Columna '{col}': {sentinel_count_orders} sentinels encontrados")
        print(f"Valores '{col}': {orders[col][orders[col].isin(sentinels_orders)].unique()}")
    else:
        print(f"Columna '{col}': sin sentinels detectados")

#correcciones para el dataset
#Monto descuento es de tipo 0, en este caso esta bien porque quiere decir que no hay descuento
#Cantidad -1 es un error el cual se debe eliminar. o calcularlo bien desde el (montotal-descuento)/precio unitario
#PRecio unitario y monto total estan bien

Columna 'cantidad': 3 sentinels encontrados
Valores 'cantidad': [-1.]
Columna 'precio_unitario': sin sentinels detectados
Columna 'monto_descuento': 12551 sentinels encontrados
Valores 'monto_descuento': [0.]
Columna 'monto_total': sin sentinels detectados


In [13]:
#filtro porque son solo 3 filas de error:
orders = orders[orders['cantidad'] >= 0]

In [14]:
#revisar variables numericas no tenga negativos para Orders
var_numerica_m =['gasto']
marketing[var_numerica_m].describe()

#puedo encontrar sentinels en las variables numericas?
sentinels_marketing = [-999, 0.0000, -1, 9999, "NA", "?", "NULL", "Unknown", "", " "]

for col in var_numerica_m:
    sentinel_count_maketing = marketing[col].isin(sentinels_marketing).sum()
    if sentinel_count_maketing > 0:
        print(f"Columna '{col}': {sentinel_count_marketing} sentinels encontrados")
        print(f"Valores '{col}': {marketing[col][marketing[col].isin(sentinels_marketing)].unique()}")
    else:
        print(f"Columna '{col}': sin sentinels detectados")


#esta bien el data set

Columna 'gasto': sin sentinels detectados


In [15]:
#cantidad de nulos
print("Nulos de Orders")
print(orders.isna().sum()/25100*100)
# pais es quien mas nulos tiene, que representa el 1%, esto me dice que puedo eliminarlos
print("Nulos de Marketing")
print(marketing.isna().sum()/1620*100)
#canal tiene el 6% de nulos, esto me dice que puedo eliminarlos

Nulos de Orders
id_pedido             0.000000
id_usuario            0.000000
fecha_hora_pedido     0.000000
pais                  1.179283
dispositivo           0.079681
fuente_referencia     0.119522
nombre_producto       0.119522
categoria_producto    0.119522
cantidad              0.000000
precio_unitario       0.000000
monto_descuento       0.000000
monto_total           0.000000
dtype: float64
Nulos de Marketing
fecha         0.000000
pais          0.000000
id_campaña    0.000000
canal         6.234568
gasto         0.000000
dtype: float64


In [16]:
#Verificar consistencia de montos
orders['check_monto'] = orders['cantidad'] * orders['precio_unitario'] - orders['monto_descuento']
inconsistencias = orders[orders['monto_total'] != orders['check_monto']]
print("Inconsistencias detectadas:", inconsistencias.shape[0])

Inconsistencias detectadas: 6592


In [17]:
#detectar valores nulos en pais
orders['pais'].isna().sum()


296

In [18]:
#eliminar los valores nulos en pais
orders = orders.dropna(subset=['pais'])
orders['pais'].isna().sum()

0

In [19]:
#detectar valores nulos en pais
orders[['dispositivo','fuente_referencia','nombre_producto','categoria_producto']].isna().sum()


dispositivo           20
fuente_referencia     30
nombre_producto       30
categoria_producto    30
dtype: int64

In [20]:
#eliminar los valores nulos en pais
orders = orders.dropna(subset=['dispositivo','fuente_referencia','nombre_producto','categoria_producto'])
orders[['dispositivo','fuente_referencia','nombre_producto','categoria_producto']].isna().sum()

dispositivo           0
fuente_referencia     0
nombre_producto       0
categoria_producto    0
dtype: int64

In [21]:
#detectar valores nulos en canal
marketing['canal'].isna().sum()


101

In [22]:
#eliminar los valores nulos en canal
marketing = marketing.dropna(subset=['canal'])
marketing['canal'].isna().sum()

0

In [23]:
#Eliminar duplicados 
orders = orders.drop_duplicates()
orders.info()
#sefiltran y se eliminan duplicados

<class 'pandas.core.frame.DataFrame'>
Int64Index: 24600 entries, 0 to 24999
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           24600 non-null  object        
 1   id_usuario          24600 non-null  object        
 2   fecha_hora_pedido   24600 non-null  datetime64[ns]
 3   pais                24600 non-null  object        
 4   dispositivo         24600 non-null  object        
 5   fuente_referencia   24600 non-null  object        
 6   nombre_producto     24600 non-null  object        
 7   categoria_producto  24600 non-null  object        
 8   cantidad            24600 non-null  float64       
 9   precio_unitario     24600 non-null  float64       
 10  monto_descuento     24600 non-null  float64       
 11  monto_total         24600 non-null  float64       
 12  check_monto         24600 non-null  float64       
dtypes: datetime64[ns](1), float64(5), object(7)
me

In [24]:
#Eliminar duplicados en Marketing
marketing = marketing.drop_duplicates()
marketing.info()
#no se encontraron duplicados, eran solo los nulos de canal.

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1519 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   fecha       1519 non-null   datetime64[ns]
 1   pais        1519 non-null   object        
 2   id_campaña  1519 non-null   object        
 3   canal       1519 non-null   object        
 4   gasto       1519 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(3)
memory usage: 71.2+ KB


In [25]:
#Revisar las variables categoricas, que esten bien los nombres
# Pasar todo a minúsculas y quitar espacios
orders['pais'] = orders['pais'].str.strip().str.lower()
orders['pais'].value_counts()
    

mexico       8305
colombia     8273
argentina    8022
Name: pais, dtype: int64

In [26]:
#Revisar las variables categoricas, que esten bien los nombres
# En este caso estan bien al igual que las de catalogo, coinciden.

orders['nombre_producto'].value_counts()
    

Jacket-Winter-M         4122
Vacuum-Pro-Black        4114
Blender-XL-Red          4114
Sneakers-Urban-42       4072
Laptop-Gaming-16GB      2752
Tablet-Standard-64GB    2729
Phone-Pro-128GB         2697
Name: nombre_producto, dtype: int64

In [27]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? 
- ¿Cuál es el costo total? 
- ¿Cuánto se ha invertido en marketing? 
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? 
- ¿Cuál es la cantidad promedio de productos por orden? 
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal? 

# Rentabilidad

In [28]:
#'el monto_total' ya es el ingreso por orden
ingreso_total = orders['monto_total'].sum()
print("Ingreso total:", ingreso_total)

Ingreso total: 51836375.13999999


In [29]:
#El costo total, sera la multiplicacion de la cantidad por orden y el costo unitario de cada produto
#debo hacer un merge para crearla medida de costo total

orders_catalog = orders.merge(catalog, on='nombre_producto', how='left')
orders_catalog['costo_total'] = orders_catalog['cantidad'] * orders_catalog['costo_unitario']
costo_total = orders_catalog['costo_total'].sum()
print("Costo total:", costo_total)

Costo total: 43078678.8


In [30]:
#inversion en marketing
marketing_total = marketing['gasto'].sum()
print("Marketing total:", marketing_total)

Marketing total: 2694664.4299999997


In [31]:
#Profit o rentabilidad del negocio
profit = ingreso_total - costo_total - marketing_total
print("Profit:", profit)

Profit: 6063031.909999996


# Ventas

In [32]:
#Ticket promedio de ordenes generadas
ticket_promedio = orders['monto_total'].mean()
print("Ticket promedio:", ticket_promedio)

Ticket promedio: 2107.169721138211


In [33]:
#cantidad promedio de ordenes por producto generadas
cantidad_promedio = orders['cantidad'].mean()
print("Cantidad promedio de productos:", cantidad_promedio)

Cantidad promedio de productos: 7.195650406504065


In [34]:
#Produco mas vendido
producto_mas_vendido = orders.groupby('nombre_producto')['cantidad'].sum().idxmax()
print("Producto más vendido:", producto_mas_vendido)

Producto más vendido: Laptop-Gaming-16GB


In [35]:
#Gasto en marketing por canal
gasto_por_canal = marketing.groupby('canal')['gasto'].sum()
print(gasto_por_canal)

canal
organic        913533.01
paid_search    863088.21
social         918043.21
Name: gasto, dtype: float64


---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [36]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [37]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()
#nombre evento me servirá para el funnel

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [38]:
#Ver como seria la secuencia del embudo.
query_events = '''
SELECT DISTINCT nombre_evento
FROM events
ORDER BY nombre_evento;
'''

events_unique = pd.read_sql(query_events, con=engine)
events_unique

,nombre_evento
0,add_payment_info
1,add_to_cart
2,begin_checkout
3,first_visit
4,purchase
5,select_item


In [39]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
SELECT 
    nombre_evento,
    COUNT(DISTINCT id_usuario) AS usuarios_unicos
FROM events
GROUP BY nombre_evento
ORDER BY 
    CASE nombre_evento
       WHEN 'first_visit' THEN 1
        WHEN 'select_item' THEN 2
        WHEN 'add_to_cart' THEN 3
        WHEN 'begin_checkout' THEN 4
        WHEN 'add_payment_info' THEN 5
        WHEN 'purchase' THEN 6
        ELSE 99
    END;
'''

totals = pd.read_sql(query_totals, con=engine)
totals

#se ve una disminución por cada una de las etapas, aunque no es muy significativa
#al final muchos compran.

,nombre_evento,usuarios_unicos
0,first_visit,7796
1,select_item,7582
2,add_to_cart,7634
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


In [40]:
# PARTE 2: Conversiones
query_conversion = '''
--1) Crear una CTE por cada evento del funnel
WITH first_visit AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'first_visit'
),
select_item AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'select_item'
),
add_to_cart AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'add_to_cart'
),
begin_checkout AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'begin_checkout'
),
add_payment_info AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'add_payment_info'
),
purchase AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'purchase'
),
-- 2) Unir las CTEs anclando en first_visit y contar usuarios por etapa
funnel_counts AS (
    SELECT
        COUNT(DISTINCT fv.id_usuario) AS usuarios_first_visit,
        COUNT(DISTINCT si.id_usuario) AS usuarios_select_item,
        COUNT(DISTINCT ac.id_usuario) AS usuarios_add_to_cart,
        COUNT(DISTINCT bc.id_usuario) AS usuarios_begin_checkout,
        COUNT(DISTINCT ap.id_usuario) AS usuarios_add_payment_info,

        COUNT(DISTINCT p.id_usuario) AS usuarios_purchase

    FROM first_visit fv
    LEFT JOIN select_item si ON fv.id_usuario = si.id_usuario
    LEFT JOIN add_to_cart ac ON fv.id_usuario = ac.id_usuario
    LEFT JOIN begin_checkout bc ON fv.id_usuario = bc.id_usuario
    LEFT JOIN add_payment_info ap ON fv.id_usuario = ap.id_usuario
    LEFT JOIN purchase p ON fv.id_usuario = p.id_usuario
)

-- 3) Calcular conversiones por etapa respecto a first_visit
SELECT
    ROUND(usuarios_select_item * 100.0 / NULLIF(usuarios_first_visit,0), 2) AS conversion_select_item,
    ROUND(usuarios_add_to_cart * 100.0 / NULLIF(usuarios_first_visit,0), 2) AS conversion_add_to_cart,
    ROUND(usuarios_begin_checkout * 100.0 / NULLIF(usuarios_first_visit,0), 2) AS conversion_begin_checkout,
    ROUND(usuarios_add_payment_info * 100.0 / NULLIF(usuarios_first_visit,0), 2) AS conversion_add_payment_info,
    ROUND(usuarios_purchase * 100.0 / NULLIF(usuarios_first_visit,0), 2) AS conversion_purchase
FROM funnel_counts;
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

,conversion_select_item,conversion_add_to_cart,conversion_begin_checkout,conversion_add_payment_info,conversion_purchase
0,94.83,95.42,90.19,78.09,77.83


---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [41]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [42]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [43]:
# Explorar tabla user_activity
print(user_activity['dias_despues_registro'].max())
print(user_activity['dias_despues_registro'].value_counts().sort_index())

28
7     8000
14    8000
21    8000
28    8000
Name: dias_despues_registro, dtype: int64


In [45]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''
-- 1) Definir cohortes por mes de registro
WITH user_cohorts AS (
    SELECT 
        id_usuario,
        CAST(fecha_registro AS DATE) AS fecha_registro,
        DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS cohorte_mes
    FROM users
),

-- 2) Identificar actividad semanal desde el registro
activity AS (
    SELECT 
        ua.id_usuario,
        CAST(ua.fecha_actividad AS DATE) AS fecha_actividad,
        ua.dias_despues_registro,
        CASE 
            WHEN ua.dias_despues_registro BETWEEN 1 AND 7 THEN 1
            WHEN ua.dias_despues_registro BETWEEN 8 AND 14 THEN 2
            WHEN ua.dias_despues_registro BETWEEN 15 AND 21 THEN 3
            WHEN ua.dias_despues_registro BETWEEN 22 AND 28 THEN 4
            ELSE NULL
        END AS semana,
        ua.activo
    FROM user_activity ua
),

-- 3) Unir cohortes con actividad
cohort_activity AS (
    SELECT 
        c.cohorte_mes,
        c.id_usuario,
        a.semana,
        a.activo
    FROM user_cohorts c
    LEFT JOIN activity a ON c.id_usuario = a.id_usuario
)

-- 4) Calcular retención semanal
SELECT 
    cohorte_mes,
    COUNT(DISTINCT id_usuario) AS usuarios_iniciales,
    COUNT(DISTINCT CASE WHEN semana = 1 AND activo = 1 THEN id_usuario END) AS retenido_w1,
    COUNT(DISTINCT CASE WHEN semana = 2 AND activo = 1 THEN id_usuario END) AS retenido_w2,
    COUNT(DISTINCT CASE WHEN semana = 3 AND activo = 1 THEN id_usuario END) AS retenido_w3,
    COUNT(DISTINCT CASE WHEN semana = 4 AND activo = 1 THEN id_usuario END) AS retenido_w4,
    ROUND(COUNT(DISTINCT CASE WHEN semana = 1 AND activo = 1 THEN id_usuario END) * 100.0 / NULLIF(COUNT(DISTINCT id_usuario),0), 2) AS semana_1,
    ROUND(COUNT(DISTINCT CASE WHEN semana = 2 AND activo = 1 THEN id_usuario END) * 100.0 / NULLIF(COUNT(DISTINCT id_usuario),0), 2) AS semana_2,
    ROUND(COUNT(DISTINCT CASE WHEN semana = 3 AND activo = 1 THEN id_usuario END) * 100.0 / NULLIF(COUNT(DISTINCT id_usuario),0), 2) AS semana_3,
    ROUND(COUNT(DISTINCT CASE WHEN semana = 4 AND activo = 1 THEN id_usuario END) * 100.0 / NULLIF(COUNT(DISTINCT id_usuario),0), 2) AS semana_4
FROM cohort_activity
GROUP BY cohorte_mes
ORDER BY cohorte_mes;
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final['cohorte_mes'] = cohorte_final['cohorte_mes'].dt.tz_localize(None)
cohorte_final

,cohorte_mes,usuarios_iniciales,retenido_w1,retenido_w2,retenido_w3,retenido_w4,semana_1,semana_2,semana_3,semana_4
0,2025-01-01,1627,697,668,656,671,42.84,41.06,40.32,41.24
1,2025-02-01,1444,611,609,635,575,42.31,42.17,43.98,39.82
2,2025-03-01,1636,677,705,690,673,41.38,43.09,42.18,41.14
3,2025-04-01,1606,680,697,663,652,42.34,43.40,41.28,40.60
4,2025-05-01,1687,695,676,706,679,41.20,40.07,41.85,40.25




---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado**  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** La tasa de conversion es igual en ambos grupos (la UI nueva no cambia la conversion)
   - **H₁ (Hipótesis alternativa):** La tasa de conversion *no* es igual en ambos grupos (la UI nueva si impacta la la conversion)
   
**Test estadístico:** el Z-test es el indicado para proporciones de conversion.  
**Nivel de significancia alpha:** ...

In [46]:
# tu código aquí
Experimento = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')
Experimento.head(10)

,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12
5,exp_user_5,control,0,mobile,Mexico,206.70,2025-01-26
6,exp_user_6,control,0,desktop,Mexico,280.11,2025-04-15
7,exp_user_7,tratamiento,0,mobile,Argentina,28.47,2025-03-18
8,exp_user_8,tratamiento,0,desktop,Mexico,287.04,2025-02-05
9,exp_user_9,control,0,mobile,Argentina,258.91,2025-02-11


In [47]:
from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import levene, ttest_ind

# Contar conversiones y totales por grupo
conversions = Experimento.groupby("variante")["convirtio"].sum()
totals = Experimento.groupby("variante")["convirtio"].count()

# Z-test de dos proporciones
z_stat, p_value = proportions_ztest(count=conversions, nobs=totals)

print("Estadístico Z:", z_stat)
print("Valor p:", p_value)

Estadístico Z: -0.8132782986429474
Valor p: 0.41605851639119995


In [48]:
from scipy.stats import levene, ttest_ind
# Separar grupos
control = Experimento[Experimento['variante'] == 'control']['convirtio']
tratamiento = Experimento[Experimento['variante'] == 'tratamiento']['convirtio']

#Test de Levene para varianzas
alpha = 0.05 # umbral de significancia
l_stat, p_value_var = levene(control, tratamiento)

print("Estadístico de Levene:", l_stat)
print("Valor p:", p_value_var)

if p_value_var < alpha: 
    print("Rechazamos H0: hay evidencia de varianzas diferentes." )
    equal_var= False
else: 
    print("No rechazamos la H0: no hay evidencia de varianzas diferentes.")
    equal_var=True
        
#hacia que lado se mueven las diferencias - medias
media_control = control.mean()
media_tratamiento = tratamiento.mean()
print("Media control:", media_control)
print("Media tratamiento:", media_tratamiento)
print("Diferencia (tratamiento - control):", media_tratamiento - media_control)

#diferencia positiva, quiere decir que si mejora, un poco, pero no hay evidencia de varianzas diferentes

Estadístico de Levene: 0.661333048721088
Valor p: 0.4161090861097335
No rechazamos la H0: no hay evidencia de varianzas diferentes.
Media control: 0.1568982880161128
Media tratamiento: 0.1628599801390268
Diferencia (tratamiento - control): 0.005961692122914003


In [ ]:
#diferencia con el Test t de Student
t_stat, p_value = ttest_ind(control, tratamiento, equal_var=equal_var)
print("Estadístico t:", t_stat)
print("Valor p:", p_value)

if p_value < alpha:
    print("Rechazamos H0: la UI impacta la tasa de conversión")
else:
    print("No rechazamos H0: no hay evidencia de impacto")

#el T-tes no es mejor al alpha, entonces la nueva Ui no impacta la conversion  

<div class="alert alert-block alert-success">
<b>Comentario del revisor</b> <a class="tocSkip"></a><br>
<b>Éxito</b> - La validación estadística está bien desarrollada y la interpretación de los resultados conecta correctamente la evidencia obtenida con el impacto esperado del experimento en la conversión.
</div>


---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---

In [ ]:
# Se pide como "recomendación" crear un Drill-Through. Pedi ayudas en discoird con el datacolearning, con la IA.
#en mi visualización de PowerBi, no aparece dicha opción. intento activarla,pero no funciona.
#si entrar en información de pagina, está activo para varios informes, pero realmente no funciona cuando lo presiono.
#lo intenté.

# Comentario General del Revisor

<div class="alert alert-block alert-success">
<b>Comentario del revisor</b> <a class="tocSkip"></a><br>

¡Felicidades! Tu proyecto está <b>aprobado</b>. El trabajo desarrollado responde correctamente a las preguntas de negocio planteadas en el brief y demuestra una buena capacidad para transformar datos en hallazgos accionables para la toma de decisiones.

#### Código y análisis

<b>Preparación y calidad de datos</b>

* La limpieza de datos fue desarrollada de forma estructurada y consistente. Se validaron formatos de fecha, variables categóricas, valores nulos, duplicados y consistencia de montos antes de iniciar el análisis.
* La estandarización de categorías como países y la validación de variables numéricas fortalecen la confiabilidad del dataset final utilizado en el proyecto.

<b>Análisis de rentabilidad</b>

* El cálculo de revenue, costos, profit y gasto en marketing permite evaluar correctamente la viabilidad financiera del negocio.
* La integración entre orders y catalog para calcular costos unitarios muestra una correcta relación entre datasets y una buena comprensión del modelo de negocio.
* El análisis identifica claramente que el revenue está altamente concentrado en Laptop-Gaming-16GB, aportando contexto importante para interpretar la rentabilidad.

<b>Análisis de comportamiento de ventas</b>

* Las métricas de ticket promedio, cantidad promedio de productos y producto más vendido fueron correctamente calculadas e interpretadas.
* El análisis por canal de marketing aporta contexto útil sobre la distribución del gasto y el comportamiento comercial.

<b>Funnel de conversión</b>

* La construcción del funnel está correctamente organizada y permite visualizar claramente el comportamiento de los usuarios en cada etapa del proceso de compra.
* El cálculo de conversiones aporta una lectura clara sobre la eficiencia del flujo de checkout y la pérdida de usuarios entre etapas.

<b>Retención por cohortes</b>

* El análisis de cohortes está bien planteado y permite medir correctamente la retención semanal de usuarios desde el registro.
* La utilización de cohortes mensuales y porcentajes de retención facilita comparar estabilidad y comportamiento entre grupos de usuarios.

<b>Validación estadística</b>

* El experimento A/B fue correctamente desarrollado utilizando un Z-test para proporciones, adecuado para evaluar diferencias en tasas de conversión.
* La interpretación estadística está alineada con los resultados obtenidos y permite concluir correctamente que no existe evidencia suficiente para afirmar que la nueva UI mejora la conversión.
* El análisis adicional mediante Levene y T-test aporta profundidad al razonamiento estadístico.

<b>Interpretación de negocio</b>

* Las conclusiones conectan adecuadamente los hallazgos técnicos con implicaciones reales para el negocio.
* El proyecto logra identificar problemas potenciales relacionados con costos, concentración de revenue y eficiencia del marketing.



#### Dashboard y visualización

<b>Presentación de KPIs</b>

* Los indicadores principales están correctamente priorizados y permiten interpretar rápidamente el estado financiero y operativo del negocio.
* La inclusión de revenue, profit, ticket promedio, marketing y promedio de productos vendidos facilita una lectura ejecutiva clara.

<b>Análisis temporal</b>

* Las gráficas mensuales permiten detectar variaciones relevantes en ingresos, profit y comportamiento comercial a lo largo del tiempo.
* La comparación entre revenue, profit y gasto de marketing ayuda a interpretar la evolución del negocio y su rentabilidad.

<b>Análisis de productos y categorías</b>

* Las visualizaciones muestran claramente la alta concentración de ventas en Laptop-Gaming-16GB y permiten comparar revenue y profit entre categorías.
* La tabla resumen aporta valor analítico adicional al incluir métricas como margen, órdenes, ROI y costos por producto.

<b>Segmentación y exploración interactiva</b>

* La incorporación de filtros por país, mes y producto mejora significativamente la exploración del dashboard y facilita análisis más específicos.
* La segmentación permite comparar comportamiento comercial y gasto de marketing entre distintos contextos.

<b>Diseño visual y comunicación</b>

* La distribución de elementos es clara y organizada, permitiendo navegar correctamente entre métricas y visualizaciones.
* El dashboard comunica adecuadamente los principales hallazgos del proyecto y complementa de manera efectiva el análisis técnico realizado.

El proyecto demuestra un buen manejo de análisis de datos, interpretación de métricas de negocio y comunicación visual de resultados.

</div>
